# 03 — A0 Evaluation

Fixed project root: `D:\PAPERS\SPEECH\low_snr_speech_enhancement`

Default target: VoiceBank+DEMAND official test manifest.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from pesq import pesq
from pystoi import stoi

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")
SR = 16000
N_FFT = 512
WIN_LENGTH = 400
HOP_LENGTH = 100

CHECKPOINT = PROJECT_ROOT / "outputs" / "a0_voicebank" / "best.pt"
TEST_MANIFEST = PROJECT_ROOT / "manifests" / "voicebank" / "test.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "a0_voicebank" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT.exists(), f"Checkpoint missing: {CHECKPOINT}"
assert TEST_MANIFEST.exists(), f"Test manifest missing: {TEST_MANIFEST}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
def read_audio(path, target_sr=16000):
    wav, sr = sf.read(path, always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr).astype(np.float32)
    return wav

def stft_complex(waveform):
    window = torch.hann_window(WIN_LENGTH, device=waveform.device)
    return torch.stft(waveform, n_fft=N_FFT, hop_length=HOP_LENGTH,
                      win_length=WIN_LENGTH, window=window,
                      center=True, return_complex=True)

def istft_complex(spec, length):
    window = torch.hann_window(WIN_LENGTH, device=spec.device)
    return torch.istft(spec, n_fft=N_FFT, hop_length=HOP_LENGTH,
                       win_length=WIN_LENGTH, window=window,
                       center=True, length=length)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class MagnitudeUNet(nn.Module):
    def __init__(self, base_channels=16, depth=4):
        super().__init__()
        channels = [base_channels*(2**i) for i in range(depth)]
        self.encoders = nn.ModuleList()
        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            in_ch = ch
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(channels[-1], channels[-1]*2)
        self.up_convs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        dec_in = channels[-1]*2
        for ch in reversed(channels):
            self.up_convs.append(nn.Conv2d(dec_in, ch, 1))
            self.decoders.append(ConvBlock(ch*2, ch))
            dec_in = ch
        self.out = nn.Conv2d(channels[0], 1, 1)

    def forward(self, noisy_mag):
        x = torch.log1p(noisy_mag).unsqueeze(1)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.up_convs, self.decoders, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        mask = torch.sigmoid(self.out(x)).squeeze(1)
        return mask * noisy_mag, mask

model = MagnitudeUNet().to(device)
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

print("Loaded checkpoint:", CHECKPOINT)

In [ ]:
def align(clean, estimate):
    n = min(len(clean), len(estimate))
    return np.asarray(clean[:n], dtype=np.float64), np.asarray(estimate[:n], dtype=np.float64)

def si_sdr(clean, estimate, eps=1e-8):
    clean, estimate = align(clean, estimate)
    clean -= clean.mean()
    estimate -= estimate.mean()
    scale = np.dot(estimate, clean) / (np.dot(clean, clean) + eps)
    target = scale * clean
    residual = estimate - target
    return 10*np.log10((np.sum(target**2)+eps)/(np.sum(residual**2)+eps))

def snr_ref(clean, estimate, eps=1e-8):
    clean, estimate = align(clean, estimate)
    err = estimate-clean
    return 10*np.log10((np.sum(clean**2)+eps)/(np.sum(err**2)+eps))

def compute_metrics(clean, estimate):
    clean, estimate = align(clean, estimate)
    out = {}
    try:
        out["pesq"] = float(pesq(SR, clean, estimate, "wb"))
    except Exception:
        out["pesq"] = np.nan
    try:
        out["stoi"] = float(stoi(clean, estimate, SR, extended=False))
    except Exception:
        out["stoi"] = np.nan
    try:
        out["estoi"] = float(stoi(clean, estimate, SR, extended=True))
    except Exception:
        out["estoi"] = np.nan
    out["si_sdr"] = float(si_sdr(clean, estimate))
    out["snr_ref"] = float(snr_ref(clean, estimate))
    return out

In [ ]:
df = pd.read_csv(TEST_MANIFEST)
rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating A0"):
    clean = read_audio(row["clean_path"], SR)
    noisy = read_audio(row["noisy_path"], SR)
    n = min(len(clean), len(noisy))
    clean, noisy = clean[:n], noisy[:n]

    noisy_t = torch.from_numpy(noisy).unsqueeze(0).to(device)

    with torch.no_grad():
        noisy_spec = stft_complex(noisy_t)
        enhanced_mag, _ = model(noisy_spec.abs())
        enhanced_spec = enhanced_mag * torch.exp(1j*torch.angle(noisy_spec))
        enhanced = istft_complex(enhanced_spec, length=n).squeeze(0).cpu().numpy()

    noisy_metrics = compute_metrics(clean, noisy)
    enh_metrics = compute_metrics(clean, enhanced)

    rows.append({
        "utt_id": row["utt_id"],
        "speaker_id": row.get("speaker_id", None),
        **{f"noisy_{k}": v for k,v in noisy_metrics.items()},
        **{f"enh_{k}": v for k,v in enh_metrics.items()},
    })

result = pd.DataFrame(rows)
result.to_csv(OUTPUT_DIR / "per_utterance.csv", index=False)
display(result.head())

In [ ]:
metric_cols = [
    "noisy_pesq", "enh_pesq",
    "noisy_stoi", "enh_stoi",
    "noisy_estoi", "enh_estoi",
    "noisy_si_sdr", "enh_si_sdr",
    "noisy_snr_ref", "enh_snr_ref"
]

summary = []
for col in metric_cols:
    vals = pd.to_numeric(result[col], errors="coerce").dropna()
    summary.append({
        "metric": col,
        "mean": vals.mean(),
        "std": vals.std(ddof=1),
        "median": vals.median(),
        "n": len(vals)
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary_df)
print("Saved evaluation outputs to:", OUTPUT_DIR)